# 04 — Narrative Testing

Tests widely-held F1 narratives against the data. Each section frames the received wisdom,
defines the measurement, then surfaces what the numbers actually say.

**Narratives in this notebook:**

1. **DNF cause categorisation** — do regulation reset years skew more mechanical than settled era years?
2. Ferrari championship bottle rate *(to be added)*
3. Safety car lottery *(to be added)*
4. Overtake index foundation *(to be added)*
5. Monaco processional racing *(to be added)*
6. DRS made overtaking artificial *(to be added)*
7. Sprints exciting before the tyre stops *(to be added)*
8. Turn 1 chaos worse in regulation reset years *(to be added)*

> **Data loading note:** The loading cell below is heavy — it iterates every race session
> across 2022–2025 via FastF1. First run populates the cache; subsequent runs are fast.
> **Do not re-run the loading cell unless you need to refresh the dataset.**

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data.fastf1_loader import load_session, get_event_schedule, get_race_control_flags
from src.utils.era_helper import get_year_within_era, get_era_name
from src.analysis.reliability import is_dnf, dnf_causes
from src.analysis.dnf_categorisation import (
    categorise_dnf, dnf_category_counts, mechanical_share_by_era_year,
    MECHANICAL, COLLISION, ADMINISTRATIVE, UNKNOWN,
)

sns.set_theme(style='whitegrid', palette='muted')
GROUND_EFFECT_SEASONS = [2022, 2023, 2024, 2025]
print('Libraries loaded.')

## Load Race Results (2022–2025)

Same loading pattern as notebook 01. Produces a flat `results_df` with one row per driver per race.
DNF status strings are preserved exactly as FastF1 returns them — categorisation happens downstream.

In [ ]:
all_results = []

_SPRINT_FORMATS = {'sprint', 'sprint_shootout', 'sprint_qualifying'}
_DNS_STATUSES = {'Did not start', 'Did not qualify', 'Withdrew'}

for season in GROUND_EFFECT_SEASONS:
    schedule = get_event_schedule(season)
    races = schedule[schedule['EventFormat'] != 'testing']

    for _, event in races.iterrows():
        round_num = int(event['RoundNumber'])
        event_name = event['EventName']
        is_sprint_wknd = event['EventFormat'] in _SPRINT_FORMATS

        try:
            session = load_session(season, round_num, 'R')
            results = session.results
            rc_flags = get_race_control_flags(session)

            for _, driver in results.iterrows():
                finish_pos = driver.get('Position')
                grid_pos = driver.get('GridPosition')
                pts = driver.get('Points')
                status = driver.get('Status', '')
                dns = status in _DNS_STATUSES
                all_results.append({
                    'season': season,
                    'round': round_num,
                    'race_name': event_name,
                    'is_sprint_weekend': is_sprint_wknd,
                    'driver_id': driver.get('Abbreviation', ''),
                    'driver_name': f"{driver.get('FirstName', '')} {driver.get('LastName', '')}".strip(),
                    'constructor_id': str(driver.get('TeamId', '')).lower().replace(' ', '_'),
                    'constructor_name': driver.get('TeamName', ''),
                    'grid_position': int(grid_pos) if pd.notna(grid_pos) and int(grid_pos) > 0 else None,
                    'finish_position': None if dns or not pd.notna(finish_pos) else int(finish_pos),
                    'points': float(pts) if pd.notna(pts) else 0.0,
                    'status': status,
                    'had_sc': rc_flags['had_sc'],
                    'had_vsc': rc_flags['had_vsc'],
                    'had_red_flag': rc_flags['had_red_flag'],
                })

            print(f'  {season} R{round_num:02d} {event_name} — {len(results)} drivers loaded')

        except Exception as e:
            print(f'  {season} R{round_num:02d} {event_name} — SKIPPED ({e})')

results_df = pd.DataFrame(all_results)

# Normalise constructor IDs across mid-era rebrands
_CONSTRUCTOR_ALIASES = {'alphatauri': 'rb', 'alfa': 'sauber'}
_CONSTRUCTOR_DISPLAY = {
    'rb':     'Racing Bulls / AlphaTauri',
    'sauber': 'Sauber / Alfa Romeo',
}
results_df['constructor_id'] = results_df['constructor_id'].replace(_CONSTRUCTOR_ALIASES)
results_df['constructor_name'] = (
    results_df['constructor_id'].map(_CONSTRUCTOR_DISPLAY).fillna(results_df['constructor_name'])
)

print(f'\nTotal race entries: {len(results_df)}')
print(f'Races: {results_df[["season","round"]].drop_duplicates().shape[0]}')

---

## 1. DNF Cause Categorisation

**The narrative:** Regulation reset years are harder on the cars — teams are running new,
unproven hardware under race stress. This should show up as a higher share of *mechanical*
DNFs in Year 1 of each era, falling in Year 2+ as reliability matures.

**The measurement:**
- Group every DNF status into four categories: Mechanical, Collision, Administrative, Unknown
- Administrative (DSQ, DNS, Withdrew) and Unknown ('Retired' with no cause) are flagged
  but excluded from the mechanical share percentage — they don't reflect car reliability
- Track mechanical share (% of attributed DNFs) by year-within-era

**Category definitions:**
| Category | Examples |
|---|---|
| Mechanical | Engine, Gearbox, Hydraulics, Power Unit, Brakes, Suspension, ERS, etc. |
| Collision | Accident, Collision, Collision damage, Spun off |
| Administrative | Disqualified, Did not start, Did not qualify, Withdrew |
| Unknown | Retired (no cause recorded) |

In [ ]:
# Isolate DNF rows
dnf_df = results_df[results_df['status'].apply(is_dnf)].copy()
print(f'Total DNFs across 2022–2025: {len(dnf_df)}')
print(f'\nRaw status frequency (top 20):')
print(dnf_causes(results_df).head(20).to_string(index=False))

In [ ]:
# Apply categorisation
dnf_df['category'] = dnf_df['status'].apply(categorise_dnf)
dnf_df['era_year'] = dnf_df['season'].apply(get_year_within_era)

# Flag any statuses landing in Unknown that are not 'Retired'
# — these are unrecognised strings that should be reviewed and mapped
unrecognised = dnf_df[(dnf_df['category'] == UNKNOWN) & (dnf_df['status'] != 'Retired')]
if not unrecognised.empty:
    print('Unrecognised statuses (review for mapping):')
    print(unrecognised['status'].value_counts().to_string())
else:
    print('All Unknown DNFs are generic Retired entries — no unrecognised statuses.')

print(f'\nDNF category totals (2022–2025):')
print(dnf_category_counts(dnf_df).to_string(index=False))

In [ ]:
# Category breakdown by season
season_counts = dnf_category_counts(dnf_df, group_by=['season'])

# Pivot for stacked bar
pivot = season_counts.pivot_table(
    index='season', columns='category', values='count', fill_value=0
).reindex(columns=[MECHANICAL, COLLISION, ADMINISTRATIVE, UNKNOWN], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar — counts
pivot.plot(kind='bar', stacked=True, ax=axes[0],
           color=['#e74c3c', '#3498db', '#95a5a6', '#bdc3c7'],
           edgecolor='white', linewidth=0.5)
axes[0].set_title('DNF Counts by Category and Season', fontsize=12)
axes[0].set_xlabel('Season')
axes[0].set_ylabel('DNF Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Category', fontsize=9)

# Stacked bar — percentages (attributed only)
attributed = pivot[[MECHANICAL, COLLISION]].copy()
attributed_pct = attributed.div(attributed.sum(axis=1), axis=0) * 100
attributed_pct.plot(kind='bar', stacked=True, ax=axes[1],
                    color=['#e74c3c', '#3498db'],
                    edgecolor='white', linewidth=0.5)
axes[1].set_title('Mechanical vs Collision Share\n(attributed DNFs only, excl. Administrative & Unknown)', fontsize=11)
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Share (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].legend(title='Category', fontsize=9)

plt.tight_layout()
plt.show()

print('\nMechanical / Collision split by season (attributed DNFs only):')
print(attributed_pct.round(1).to_string())

In [ ]:
# Mechanical share by year-within-era — the core narrative test
mech_share = mechanical_share_by_era_year(dnf_df)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    mech_share['era_year'].astype(str),
    mech_share['mechanical_pct'],
    color=sns.color_palette('Reds_r', len(mech_share)),
    edgecolor='white',
)
for bar, (_, row) in zip(bars, mech_share.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{row['mechanical_pct']:.1f}%\n(n={int(row['total_attributed'])})",
            ha='center', va='bottom', fontsize=10)

ax.set_title(
    'Mechanical DNF Share by Year-Within-Era\n'
    'Ground Effect Era (2022–2025) · Attributed DNFs only',
    fontsize=12
)
ax.set_xlabel('Year Within Era (1 = regulation reset year)')
ax.set_ylabel('Mechanical Share (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

print('\nMechanical share by year-within-era:')
print(mech_share[['era_year', MECHANICAL, COLLISION, 'total_attributed', 'mechanical_pct']].to_string(index=False))

### Findings — DNF Cause Categorisation

*(Fill in after running the cells above.)*

| Question | Finding |
|---|---|
| Does Year 1 show higher mechanical share? | *(pending)* |
| Is there a clear downward trend across era years? | *(pending)* |
| Which season had the most collision DNFs? | *(pending)* |
| What share of DNFs are 'Unknown' (Retired, no cause)? | *(pending)* |

**Narrative verdict:** *(pending — update after reviewing charts)*